<a href="https://colab.research.google.com/github/ThomasGK42/genai-notebooks/blob/main/Embeddings%20QA%20Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install --upgrade google-cloud-aiplatform

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 38.8 MB/s  0:00:00


In [2]:
%pip install -U langchain-community beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.1 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: beautifulsoup4
    Found existing installation: beautifulsoup4 4.13.5
    Uninstalling beautifulsoup4-4.13.5:
      Successfully uninstalled beautifulsoup4-4.13.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
%pip install -U langchain-text-splitters

In [4]:
%pip install -U chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 9.6 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 12.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 15.8 MB/s  0:00:01
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelemetry-sdk 1.42.1
    Uninstalling opentelemetry-sdk-1.42.1:
      Succ

In [5]:
%pip install -U google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.1 MB/s  0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [google-genai]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 

In [6]:
# Restart kernel after installs so that your environment can access the new packages
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

# Set Variables

In [1]:
# get project ID
PROJECT_ID = ! gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-central1" # @param {type:"string"}
DOCUMENT_URL = "https://www.gutenberg.org/cache/epub/55/pg55.txt" # @param {type:"string"}

# define project information manually if the above code didn't work
if PROJECT_ID == "(unset)":
  PROJECT_ID = "embeddingsqa" # @param {type:"string"}

print(PROJECT_ID)

embeddingsqa


In [2]:
import getpass
import os

# getpass will prompt for an API Key
# The API Key is needed for Chroma DB
API_KEY = getpass.getpass("Provide your Google API Key")

Provide your Google API Key··········


## Initial Vertex AI

In [3]:
# init the aiplatform package
from google.cloud import aiplatform
aiplatform.init(project=PROJECT_ID, location=LOCATION)

## Just test the Embeddings model

In [4]:
from vertexai.language_models import TextEmbeddingModel

def text_embedding(text_to_embed) -> list:
    """Text embedding with a Large Language Model."""
    model = TextEmbeddingModel.from_pretrained("gemini-embedding-001")
    embeddings = model.get_embeddings([text_to_embed])
    for embedding in embeddings:
        vector = embedding.values
        print(f"Length of Embedding Vector: {len(vector)}")
    return vector

In [5]:
from google.colab import auth
auth.authenticate_user()

In [6]:
emb1 = text_embedding("Hello World")
print(emb1)

/usr/local/lib/python3.13/dist-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Length of Embedding Vector: 3072
[-0.015046939253807068, 0.007224771194159985, 0.010408386588096619, -0.0641695111989975, -0.00329646491445601, 0.00021087047934997827, -0.013788601383566856, 0.012758294120430946, 0.0069551351480185986, 0.0027825161814689636, -0.0127637954428792, -0.021246222779154778, 0.03036157786846161, 0.047764647752046585, 0.10964751988649368, 0.014141391031444073, -0.005493228789418936, -0.012254291214048862, 0.01639988273382187, -0.014347841963171959, 0.0016168602742254734, -0.007848015055060387, 0.01726933754980564, 0.013009180314838886, -0.032668691128492355, 0.0032125739380717278, 0.020221903920173645, 0.002008603187277913, 0.03255552798509598, 0.018311819061636925, 0.02067670412361622, -0.011489714495837688, -0.026783134788274765, 0.007460409309715033, -0.000754068314563483, 0.016881272196769714, 0.009227439761161804, 0.00026535248616710305, -0.016462480649352074, -0.007008027285337448, -0.0033590917009860277, 0.00942162424325943, -0.009593556635081768, -0.00

In [7]:
from vertexai.language_models import TextEmbeddingInput

emd_with_task_type  = TextEmbeddingInput(
    text="Hello World",
    task_type="RETRIEVAL_QUERY"
)

emb2 = text_embedding(emd_with_task_type)
print(emb2)


emd_with_task_type  = TextEmbeddingInput(
    text="Hello World",
    task_type="RETRIEVAL_DOCUMENT"
)

emb3 = text_embedding(emd_with_task_type)
print(emb3)

Length of Embedding Vector: 3072
[-0.015046939253807068, 0.007224771194159985, 0.010408386588096619, -0.0641695111989975, -0.00329646491445601, 0.00021087047934997827, -0.013788601383566856, 0.012758294120430946, 0.0069551351480185986, 0.0027825161814689636, -0.0127637954428792, -0.021246222779154778, 0.03036157786846161, 0.047764647752046585, 0.10964751988649368, 0.014141391031444073, -0.005493228789418936, -0.012254291214048862, 0.01639988273382187, -0.014347841963171959, 0.0016168602742254734, -0.007848015055060387, 0.01726933754980564, 0.013009180314838886, -0.032668691128492355, 0.0032125739380717278, 0.020221903920173645, 0.002008603187277913, 0.03255552798509598, 0.018311819061636925, 0.02067670412361622, -0.011489714495837688, -0.026783134788274765, 0.007460409309715033, -0.000754068314563483, 0.016881272196769714, 0.009227439761161804, 0.00026535248616710305, -0.016462480649352074, -0.007008027285337448, -0.0033590917009860277, 0.00942162424325943, -0.009593556635081768, -0.00

In [8]:
print(emb1 == emb2)
print(emb2 == emb3)

True
False


## Load the document that you want to analyze with the LLM

In [9]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(DOCUMENT_URL)

data = loader.load()
document = data[0].page_content

# Print first 50 characters
print(document[:50])

/tmp/ipykernel_4377/3467848210.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


The Project Gutenberg eBook of The Wonderful Wizar


## The document needs to be split into chucks.

Chunk size and chunk overlap are the interesting variables. Each chunk will be represented by an embedding. Larger chunks will mean there is more data for the LLM to analyze. Smaller chunks will mean less data will be passed to the LLM at inference time.

Smaller chunks also mean more embeddings.

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1500
CHUNK_OVERLAP = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.create_documents([document])

# Convert chunks into list of strings
chunks = [chunk.page_content for chunk in chunks]

print(len(chunks))
print(chunks[10][:50])

178
Dorothy looked, and gave a little cry of fright. T


## Generate the embeddings in batches

Below are helper functions that are used to rate limit and batch the geberation of the embeddings.

In [11]:
from typing import Generator, List, Optional, Tuple
import functools
import time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from tqdm import tqdm
import math


# Define an embedding method that uses the model
def encode_texts_to_embeddings(chunks: List[str]) -> List[Optional[List[float]]]:
    try:
        model = TextEmbeddingModel.from_pretrained("gemini-embedding-001")

        # convert chunks into list[TextEmbeddingInput]
        inputs = [TextEmbeddingInput(text=chunk, task_type="RETRIEVAL_DOCUMENT") for chunk in chunks]
        embeddings = model.get_embeddings(inputs)

        # You could also generate the embeddings without the task_type.
        # Then, you are just passing a collection of strings. In a real app
        # test it multiple ways.
        # The alternative would be as follows
        # embeddings = model.get_embeddings(chunks)

        return [embedding.values for embedding in embeddings]
    except Exception as e:
        print(f"Embedding error: {type(e).__name__}: {e}")
        return [None for _ in range(len(chunks))]


# Generator function to yield batches of descriptions
def generate_batches(
    chunks: List[str], batch_size: int
) -> Generator[List[str], None, None]:
    for i in range(0, len(chunks), batch_size):
        yield chunks[i : i + batch_size]


def encode_text_to_embedding_batched(
    chunks: List[str], api_calls_per_minute: int = 20, batch_size: int = 5
) -> Tuple[List[bool], np.ndarray]:

    embeddings_list: List[List[float]] = []

    # Prepare the batches using a generator
    batches = generate_batches(chunks, batch_size)

    seconds_per_job = 60 / api_calls_per_minute

    with ThreadPoolExecutor() as executor:
        futures = []
        for batch in tqdm(
            batches, total=math.ceil(len(chunks) / batch_size), position=0
        ):
            futures.append(
                executor.submit(functools.partial(encode_texts_to_embeddings), batch)
            )
            time.sleep(seconds_per_job)

        for future in futures:
            embeddings_list.extend(future.result())

    is_successful = [
        embedding is not None for sentence, embedding in zip(chunks, embeddings_list)
    ]
    embeddings_list_successful = np.squeeze(
        np.stack([embedding for embedding in embeddings_list if embedding is not None])
    )
    return is_successful, embeddings_list_successful


In [12]:
embeddings_response = encode_text_to_embedding_batched(chunks, api_calls_per_minute=20)

  0%|          | 0/36 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
100%|██████████| 36/36 [01:48<00:00,  3.00s/it]


In [13]:
# Need the IDs for the Vector database collection
# Just use counters
ids = [str(i) for i in range(len(chunks))]

# chunks has the text, embeddings_response has the embeddings
print(chunks[10][:50])
print(embeddings_response[1][10][:10])

# All the collections need the same number of items.
print(len(chunks))
print(len(embeddings_response[1]))
print(len(ids))


Dorothy looked, and gave a little cry of fright. T
[-0.01573729  0.00325682 -0.0172877  -0.05701111  0.00135395  0.01621891
  0.00144049  0.0229775  -0.01986131  0.01578269]
178
178
178


# Chroma DB is an open source database

Google Vector Search may be a better production solution, but Chroma DB is free. So, it makes for a better demo.

In [14]:
import chromadb

chroma_client = chromadb.Client()

# Make sure the collection does not exist
try:
  chroma_client.delete_collection("document-collection")
except Exception:
  pass

# Create the collection
chroma_collection = chroma_client.create_collection("document-collection", embedding_function=None)

# Add the data to the collection
chroma_collection.add(ids=ids, documents=chunks, embeddings=embeddings_response[1].tolist())
print("Documents stored:", chroma_collection.count())


Documents stored: 178


## Helper functions

In [15]:
import vertexai
from vertexai.preview.generative_models import GenerativeModel, Part
from vertexai.language_models import TextEmbeddingModel
from google import genai
from google.genai.types import HttpOptions

# Initialize the embedding model
embedding_model = TextEmbeddingModel.from_pretrained("gemini-embedding-001")

# Prompt template just uses a string template to
# format the promopt for the LLM
prompt_template = """

Answer the users question using the following data.
Only use the data provided below. Do not make anything up
or use your own data.

Data: {0}

Question: {1}

Answer:
"""

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location="global",
    http_options=HttpOptions(api_version="v1")
)

# Send the prompt to Gemini
def generate(prompt):
  response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "max_output_tokens": 2048,
            "temperature": 0.2,
            "top_p": 1
        }
    )
  return response.text

# The question is converted to an embedding.
# Then, that embedding is used to query the vector DB.
# the 5 closest embeddings are returned.
def query_vector_db(question):
  query_embedding = embedding_model.get_embeddings([question])[0].values

  results = chroma_collection.query(query_embeddings=[query_embedding], n_results=5)
  retrieved_documents = results['documents'][0]

  # concatentate all the strings in the retrieved_documents collection
  DATA = " ".join(retrieved_documents)
  return DATA

def ask_question(question):
  DATA = query_vector_db(question)
  #print(DATA)
  prompt = prompt_template.format(DATA, question)
  answer = generate(prompt)
  return answer



/usr/local/lib/python3.13/dist-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [16]:
from IPython.display import display

QUESTION1 = "Who wrote the Wizard of Oz?"
QUESTION2 = "Who were the main characters in the Wizard of Oz?"
QUESTION3 = "What was the plot of the Wizard of Oz?"
QUESTION4 = "Tell me about the Scarecrow?"

questions = [QUESTION1, QUESTION2, QUESTION3, QUESTION4]

for question in questions:
  answer = ask_question(question)
  display(answer)
  print("-----------------------")


'L. Frank Baum wrote The Wonderful Wizard of Oz.'

-----------------------


'The main characters mentioned in the provided text are Dorothy, the Scarecrow, the Tin Woodman, the Cowardly Lion, and Oz (the Great Wizard).'

-----------------------


'The plot involves Dorothy wanting to get back to her aunt and uncle in Kansas. She is told that Oz, the Great Wizard who lives in the City of Emeralds, can help her. The Scarecrow wants brains, the Tin Woodman wants a heart, and the Cowardly Lion wants courage from Oz. Oz is described as being more powerful than all the other witches and wizards together, and he can take on any form he wishes. When Dorothy finally speaks to Oz, he tells her that she must kill the Wicked Witch of the West in return for him sending her back to Kansas. Dorothy had previously killed the Witch of the East and wears her silver shoes, which bear a powerful charm.'

-----------------------


'The Scarecrow is a figure whose head is a small sack stuffed with straw, with painted eyes, nose, and mouth. He wears an old, pointed blue hat and a worn, faded blue suit of clothes, also stuffed with straw, with old boots on his feet. He was initially placed high on a pole in a cornfield to keep birds away.\n\nHe can wink, nod his head, speak, bow, walk, stretch, and yawn. He finds it tedious to be perched on the pole and asks Dorothy to take him down. He feels sad because he is not a good Scarecrow, as birds are not afraid of him; an old crow even perched on his shoulder and said he was "only stuffed with straw."\n\nThe Scarecrow believes he has no brains because he is stuffed with straw and finds it an "uncomfortable feeling to know one is a fool." He desires to have brains, believing they are "the only things worth having in this world," and hopes the Great Oz will give them to him so he can become "as much a man as any other." He doesn\'t mind his body being stuffed because he ca

-----------------------
